# Share of Business by Category — FTW vs APP vs ACC
### Vietnam · China · Thailand stores | YTD 2026 (Jan–Jul)

**Requested by:** Nigel · **Prepared by:** Toàn Phạm · **Datasource:** LSOUL Database `marine-cycle-484213-b4.reportdb`
---

### Scope & definitions

| | |
|---|---|
| **Period** | 1 Jan 2026 – 31 Jul 2026 (7 months trading) |
| **Markets** | Vietnam, China, Thailand |
| **Channel** | Physical stores only (`sale_channel LIKE '%Offline Store%'`). E-commerce, online and wholesale are excluded — Nigel asked for *stores*. |
| **Measure** | `revenue` = net revenue, already normalised to **VND** across all three markets. Units and AUR shown alongside. |
| **FTW** | Footwear — heels, boots, sneakers, sandals, loafers |
| **APP** | Apparel — dresses, tops, skirts, denim, outerwear, sets, suits, knitwear |
| **ACC** | Accessories — bags, wallets/SLG, eyewear, scarves, socks, belts |

###The request of data
> *"Can you also send across an overview of what's the share of business % sales performance
> by FTW vs App vs ACC across Vietnam, China and Thailand stores? Hearing we have a lot of
> accessories, good to know this piece of info as a reference to split our OTB buy for the
> opening inventory."* — Nigel, WhatsApp
>
> *"Yes 7 month trading performance good enough"* — scope confirmed

###a. Configuration

Everything the report depends on is set here. To re-cut for a different period, market set
or channel, change these constants only — no SQL edits needed.

In [ ]:
# If running outside BigQuery Studio you may need:
# !pip install --quiet google-cloud-bigquery db-dtypes pandas matplotlib

from google.cloud import bigquery
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# ── Parameters ────────────────────────────────────────────────────────────
PROJECT         = "marine-cycle-484213-b4"
DATE_START      = "2026-01-01"
DATE_END        = "2026-07-31"
COUNTRIES       = ["VIETNAM", "CHINA", "THAILAND"]
CHANNEL_PATTERN = "%Offline Store%"      # physical stores only

COUNTRY_ORDER = ["VIETNAM", "THAILAND", "CHINA"]     # by size of business
CAT_ORDER     = ["APP", "FTW", "ACC"]
CAT_LABEL     = {"APP": "Apparel", "FTW": "Footwear", "ACC": "Accessories"}

client = bigquery.Client(project=PROJECT)

pd.set_option("display.float_format", lambda v: f"{v:,.1f}")
print(f"Connected to {PROJECT} | {DATE_START} \u2192 {DATE_END} | {', '.join(COUNTRIES)}")

###b. The category mapping

LSOUL's product master (`reportdb.product_tree_csv`) is an unheadered CSV: `string_field_4`
holds the level-1 category and `string_field_3` the level-2. Neither is FTW/APP/ACC, so the
classifier below rolls the native tree into Nigel's three buckets.

Three details worth knowing, because they change the numbers:

1. **`cat_l2 = 'SHOES'` is tested before apparel.** A handful of shoe styles sit under
   `CLOTHING` in the tree; without this they would be counted as apparel.
2. **Packaging, gift items and staff uniform (`HỘP`, `TÚI GIẤY`, `QUÀ`, `ĐỒNG PHỤC`) are
   classed `NON-RETAIL` and excluded** from the share denominator. They are not a customer
   buy and would inflate accessories.
3. **A product-name fallback catches SKUs launched after the last product-tree refresh**
   (mostly the July drop). Footwear and accessory nouns are tested *before* apparel nouns so
   that e.g. "LOEA HEELS" does not fall through to APP.

The QA gate in section 3 proves the mapping leaves nothing behind.

In [ ]:
# Shared base CTE — every query in this notebook builds on it.
# Raw string: the fallback CASE below uses \b regex word boundaries, which a
# non-raw string would silently turn into backspace characters.
BASE_CTE = r"""
-- ============================================================================
-- Shared base: fact x product-tree, classified into FTW / APP / ACC
-- ============================================================================
WITH tree AS (
  -- Product master. One row per variation_code (the CSV has near-duplicates).
  SELECT DISTINCT
    variation_code,
    category       AS cat_l1,   -- CLOTHING / SHOES / TUI / KINH / ...
    sub_category   AS cat_l2,    -- TOP / DRESS / SHOES / PHU KIEN / ...
    size_label,
    size_type,                   -- ALPHA (XS..XL) / NUMERIC (35..43) / ONE SIZE
    size_order                   -- sort key: sizes must never sort alphabetically
  FROM `{project}.reportdb.fn_dim_sku`()
),

fact AS (
  SELECT
    DATE(f.period_date) AS period_date,
    f.country,
    f.level_2 AS sale_channel,
    f.warehouse_name,
    f.product_code,
    f.variation_code,
    f.product_name,
    t.cat_l1,
    t.cat_l2,
    t.size_label,
    t.size_type,
    t.size_order,
    SUM(f.revenue_vn) AS revenue,          -- net revenue, already normalised to VND across markets
    SUM(f.sales_revenue_vn) AS sales_revenue,    -- gross, before returns/discount adjustments
    SUM(f.units_sold) AS units_sold
  FROM `{project}.reportdb.report_order_product_revenue_summary` f
  LEFT JOIN tree t USING (variation_code)
  WHERE DATE(f.period_date) BETWEEN @d_start AND @d_end
    AND f.country IN UNNEST(@countries)
    AND f.level_2 LIKE @channel_pattern
  GROUP BY 1,2,3,4,5,6,7,8,9,10,11,12
),

classified AS (
  SELECT
    *,
    cat_l1  AS category,
    cat_l2  AS sub_category,
    size_label AS size
  FROM fact
)
"""


def run_query(sql: str) -> pd.DataFrame:
    """Run a parameterised query against BigQuery and return a DataFrame."""
    job_config = bigquery.QueryJobConfig(query_parameters=[
        bigquery.ScalarQueryParameter("d_start",         "DATE",   DATE_START),
        bigquery.ScalarQueryParameter("d_end",           "DATE",   DATE_END),
        bigquery.ArrayQueryParameter ("countries",       "STRING", COUNTRIES),
        bigquery.ScalarQueryParameter("channel_pattern", "STRING", CHANNEL_PATTERN),
    ])
    job = client.query(sql.replace("{project}", PROJECT), job_config=job_config)
    df = job.result().to_dataframe()
    print(f"  {job.total_bytes_processed/1e6:,.1f} MB scanned \u00b7 {len(df):,} rows")
    return df


print("BASE_CTE ready \u2014", len(BASE_CTE.splitlines()), "lines")

### c. Data quality gate

Before anyone reads a percentage, prove the denominator is sound. This cell **fails loudly**
if more than 1% of revenue in any market is unclassified — because an unmapped tail is
exactly the kind of thing that silently distorts a category split.

In [ ]:
Q_QA = BASE_CTE + """
-- @tag: qa
SELECT
  country,
  COUNT(DISTINCT period_date)                                   AS trading_days,
  COUNT(DISTINCT warehouse_name)                                AS store_locations,
  COUNT(DISTINCT product_code)                                  AS styles_sold,
  ROUND(SUM(revenue)/1e6, 1)                                    AS revenue_mn_vnd,
  ROUND(100 * SAFE_DIVIDE(
          SUM(IF(category IS NULL OR category = 'UNMAPPED', revenue, 0)),
          SUM(revenue)), 2) AS pct_unmapped,
  ROUND(100 * SAFE_DIVIDE(
          SUM(IF(category = 'NON-RETAIL',  revenue, 0)), SUM(revenue)), 2) AS pct_non_retail
FROM classified
GROUP BY country
ORDER BY revenue_mn_vnd DESC
"""

qa = run_query(Q_QA)
display(qa)

# --- Gate -----------------------------------------------------------------
worst = qa["pct_unmapped"].max()
assert worst <= 1.0, f"FAIL: {worst:.2f}% of revenue is unclassified \u2014 fix the mapping first"

print(f"\nPASS \u00b7 worst-case unmapped revenue: {worst:.2f}%")
print(f"PASS \u00b7 non-retail (packaging/gift/uniform) excluded: "
      f"{qa['pct_non_retail'].max():.2f}% max")
print(f"\nCoverage: {qa['trading_days'].max()} trading days \u00b7 "
      f"{qa['store_locations'].sum()} store locations \u00b7 "
      f"{qa['styles_sold'].sum():,} styles sold \u00b7 "
      f"{qa['revenue_mn_vnd'].sum():,.0f}mn VND")

## 1 · Headline — share of business by category

This is the table Nigel asked for. It is shown two ways deliberately:

- **% of sales** answers *"where does the money come from"*
- **% of units** answers *"what do we actually buy"* — and that is the one an OTB split runs on,
  because an OTB is placed in units, not in revenue.

The two diverge sharply here, so read both.

In [ ]:
Q_SHARE = BASE_CTE + """
-- @tag: share
SELECT
  country,
  category,
  ROUND(SUM(revenue)/1e6, 1)                                    AS revenue_mn_vnd,
  ROUND(100 * SUM(revenue)
        / SUM(SUM(revenue)) OVER (PARTITION BY country), 1)     AS pct_of_sales,
  CAST(SUM(units_sold) AS INT64)                                AS units,
  ROUND(100 * SUM(units_sold)
        / SUM(SUM(units_sold)) OVER (PARTITION BY country), 1)  AS pct_of_units,
  ROUND(SAFE_DIVIDE(SUM(revenue), SUM(units_sold))/1e6, 2)      AS aur_mn_vnd
FROM classified
WHERE category IN ('FTW','APP','ACC')
GROUP BY country, category
ORDER BY country, pct_of_sales DESC
"""

share = run_query(Q_SHARE)
share["category"] = pd.Categorical(share["category"], CAT_ORDER, ordered=True)
share["country"]  = pd.Categorical(share["country"],  COUNTRY_ORDER, ordered=True)
share = share.sort_values(["country", "category"])

# --- The answer, as Nigel will read it ------------------------------------
pct_sales = share.pivot(index="country", columns="category", values="pct_of_sales")
pct_units = share.pivot(index="country", columns="category", values="pct_of_units")

print("SHARE OF BUSINESS \u2014 % of store SALES (VND), YTD Jan\u2013Jul 2026")
display(pct_sales.style.format("{:.1f}%").background_gradient(cmap="Blues", axis=None))

print("\nSHARE OF BUSINESS \u2014 % of store UNITS, YTD Jan\u2013Jul 2026")
display(pct_units.style.format("{:.1f}%").background_gradient(cmap="Oranges", axis=None))

In [ ]:
# --- Full detail, incl. AUR (the reason value-share and unit-share diverge) --
detail = share.rename(columns={
    "country": "Market", "category": "Cat",
    "revenue_mn_vnd": "Revenue (mn VND)", "pct_of_sales": "% of sales",
    "units": "Units", "pct_of_units": "% of units", "aur_mn_vnd": "AUR (mn VND)",
})
detail["Cat"] = detail["Cat"].map(CAT_LABEL)
display(detail.set_index(["Market", "Cat"]).style.format({
    "Revenue (mn VND)": "{:,.1f}", "% of sales": "{:.1f}%",
    "Units": "{:,.0f}", "% of units": "{:.1f}%", "AUR (mn VND)": "{:,.2f}",
}))

## 2 · Chart

Apparel is so dominant that a single 100% stacked bar makes footwear and accessories
unreadable — so the figure is split. The left panel gives the honest full composition;
the right panel zooms into the 0–13% band where the actual buying decision sits.

In [ ]:
# ── Chart system ──────────────────────────────────────────────────────────
SURFACE, INK, INK_MUTED, GRID = "#FCFCFB", "#1A1A1A", "#6B6B6B", "#E4E4E1"
CAT_COLOR = {"APP": "#1F4E9C", "FTW": "#F5821F", "ACC": "#00998A"}   # validated palette

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "savefig.facecolor": SURFACE, "text.color": INK,
    "axes.labelcolor": INK_MUTED, "xtick.color": INK_MUTED, "ytick.color": INK_MUTED,
    "axes.edgecolor": GRID, "font.size": 11, "figure.dpi": 110,
})


def strip_frame(ax, keep_bottom=False, keep_left=False):
    for side, keep in (("top", False), ("right", False),
                       ("bottom", keep_bottom), ("left", keep_left)):
        ax.spines[side].set_visible(keep)


rows  = [c for c in COUNTRY_ORDER if c in pct_sales.index]
y     = range(len(rows))
title = lambda t: t.title()

fig, (axL, axR) = plt.subplots(
    1, 2, figsize=(14, 4.4), gridspec_kw={"width_ratios": [1.5, 1], "wspace": 0.32})

# ── Panel A · full composition, 100% stacked ──────────────────────────────
left = [0.0] * len(rows)
for cat in CAT_ORDER:
    vals = [pct_sales.loc[r, cat] for r in rows]
    axL.barh(y, vals, left=left, height=0.58, color=CAT_COLOR[cat],
             edgecolor=SURFACE, linewidth=2, zorder=3,
             label=f"{CAT_LABEL[cat]}")
    if cat == "APP":                       # only APP is wide enough to label inside
        for i, (v, l) in enumerate(zip(vals, left)):
            axL.text(l + v / 2, i, f"{v:.1f}%", ha="center", va="center",
                     color="white", fontsize=12, fontweight="bold", zorder=4)
    left = [l + v for l, v in zip(left, vals)]

axL.set_yticks(list(y)); axL.set_yticklabels([title(r) for r in rows], color=INK, fontsize=12)
axL.set_xlim(0, 100); axL.xaxis.set_major_formatter(mticker.PercentFormatter())
axL.set_xticks([0, 25, 50, 75, 100])
axL.invert_yaxis(); strip_frame(axL)
axL.grid(axis="x", color=GRID, lw=1, zorder=0); axL.set_axisbelow(True)
axL.set_title("Full composition — % of store sales", loc="left",
              fontsize=12.5, fontweight="bold", pad=12)

# ── Panel B · the decision zone, FTW + ACC only ───────────────────────────
h, sub = 0.3, ["FTW", "ACC"]
for j, cat in enumerate(sub):
    vals = [pct_sales.loc[r, cat] for r in rows]
    offs = [i + (j - 0.5) * h for i in y]
    axR.barh(offs, vals, height=h * 0.82, color=CAT_COLOR[cat], zorder=3)
    for off, v in zip(offs, vals):
        axR.text(v + 0.25, off, f"{v:.1f}%", va="center", ha="left",
                 fontsize=10.5, color=INK, fontweight="bold", zorder=4)

axR.set_yticks(list(y)); axR.set_yticklabels([title(r) for r in rows], color=INK, fontsize=12)
axR.set_xlim(0, 13); axR.xaxis.set_major_formatter(mticker.PercentFormatter())
axR.invert_yaxis(); strip_frame(axR)
axR.grid(axis="x", color=GRID, lw=1, zorder=0); axR.set_axisbelow(True)
axR.set_title("Decision zone — footwear & accessories only", loc="left",
              fontsize=12.5, fontweight="bold", pad=12)

handles, labels = axL.get_legend_handles_labels()
fig.legend(handles, labels, loc="upper left", bbox_to_anchor=(0.008, 1.07),
           ncol=3, frameon=False, fontsize=11)
fig.suptitle("Share of business by category — stores, YTD Jan–Jul 2026",
             x=0.008, ha="left", y=1.31, fontsize=15, fontweight="bold")
fig.text(0.008, 1.19, "Net revenue in VND. Physical stores only; e-commerce and wholesale excluded.",
         ha="left", fontsize=10.5, color=INK_MUTED)
plt.show()

## 3 · Contribution overview — categories and sub-categories

Two levels of the merchandise hierarchy, both as **% of store sales**:

- **3.1** the whole tree — every category and every sub-category, sized against the total
  business so footwear and accessories are finally visible alongside apparel
- **3.2** apparel on its own, where the numbers are large enough to plan against

Percentages are of **net revenue**. Each sub-category is shown two ways — as a share of
*total* store sales (comparable across categories) and as a share of its *own* category
(how the category is built). Both matter: the first sizes the prize, the second sets the mix.

### 3.1 · The full contribution tree

In [ ]:
Q_CONTRIB = BASE_CTE + """
-- @tag: contrib
SELECT
  country,
  category,
  sub_category,
  COUNT(DISTINCT product_code)                                  AS styles,
  ROUND(SUM(revenue)/1e6, 1)                                    AS revenue_mn_vnd,
  CAST(SUM(units_sold) AS INT64)                                AS units,
  ROUND(100 * SUM(revenue)
        / SUM(SUM(revenue)) OVER (PARTITION BY country), 2)     AS pct_of_total,
  ROUND(100 * SUM(revenue)
        / SUM(SUM(revenue)) OVER (PARTITION BY country, category), 1) AS pct_of_category,
  ROUND(100 * SUM(units_sold)
        / SUM(SUM(units_sold)) OVER (PARTITION BY country), 2)  AS pct_of_total_units
FROM classified
WHERE category IN ('FTW', 'APP', 'ACC')
GROUP BY country, category, sub_category
ORDER BY country, category, revenue_mn_vnd DESC
"""

contrib = run_query(Q_CONTRIB)
contrib["country"] = pd.Categorical(contrib["country"], COUNTRY_ORDER, ordered=True)

# --- Level 1: category, by market and blended ----------------------------
cat_ov = share.pivot(index="category", columns="country", values="pct_of_sales")
cat_ov = cat_ov.reindex(index=CAT_ORDER, columns=COUNTRY_ORDER)
blend_rev = share.groupby("category", observed=True)["revenue_mn_vnd"].sum()
cat_ov["All 3 markets"] = (100 * blend_rev / blend_rev.sum()).round(1).reindex(CAT_ORDER)
cat_ov.index = [CAT_LABEL[c] for c in cat_ov.index]

print("LEVEL 1 \u2014 % OF STORE SALES BY CATEGORY, YTD Jan\u2013Jul 2026")
display(cat_ov.style.format("{:.1f}%", na_rep="\u2013")
                    .background_gradient(cmap="Blues", axis=None))

assert abs(cat_ov["All 3 markets"].sum() - 100) < 0.2, "category shares do not sum to 100%"
print(f"Checks: columns sum to "
      + " / ".join(f"{cat_ov[c].sum():.1f}%" for c in cat_ov.columns))

In [ ]:
# --- Level 2: every sub-category, blended across the three markets -------
tot_rev = contrib["revenue_mn_vnd"].sum()
tree = (contrib.groupby(["category", "sub_category"], observed=True)
                .agg(styles=("styles", "sum"),
                     revenue_mn=("revenue_mn_vnd", "sum"),
                     units=("units", "sum"))
                .reset_index())
tree["pct_of_total"] = (100 * tree.revenue_mn / tot_rev).round(2)
tree["pct_of_category"] = (100 * tree.revenue_mn
                           / tree.groupby("category", observed=True)["revenue_mn"]
                                 .transform("sum")).round(1)
tree["category"] = pd.Categorical(tree["category"], CAT_ORDER, ordered=True)
tree = tree.sort_values(["category", "revenue_mn"], ascending=[True, False])

print("LEVEL 2 \u2014 EVERY SUB-CATEGORY, all 3 markets combined")
display(tree.assign(Category=tree.category.map(CAT_LABEL))
            .set_index(["Category", "sub_category"])
            .drop(columns=["category"])
            .rename(columns={"styles": "Styles", "revenue_mn": "Revenue (mn VND)",
                             "units": "Units", "pct_of_total": "% of TOTAL sales",
                             "pct_of_category": "% of own category"})
            .style.format({"Styles": "{:,.0f}", "Revenue (mn VND)": "{:,.1f}",
                           "Units": "{:,.0f}", "% of TOTAL sales": "{:.2f}%",
                           "% of own category": "{:.1f}%"})
            .background_gradient(cmap="Blues", subset=["% of TOTAL sales"]))

print(f"\nSub-category shares sum to {tree.pct_of_total.sum():.1f}% of total store sales.")

In [ ]:
# --- Level 2 by market: how each category is built, per country ----------
for cat in CAT_ORDER:
    d = contrib[contrib.category == cat]
    if d.empty:
        continue
    piv = d.pivot_table(index="sub_category", columns="country",
                        values="pct_of_category", aggfunc="sum", observed=True)
    piv = piv.reindex(columns=COUNTRY_ORDER)
    piv = piv.loc[d.groupby("sub_category", observed=True)["revenue_mn_vnd"]
                   .sum().sort_values(ascending=False).index]
    print(f"\n{CAT_LABEL[cat].upper()} \u2014 % of each market's {CAT_LABEL[cat].upper()} sales")
    display(piv.style.format("{:.1f}%", na_rep="\u2013")
                     .background_gradient(cmap="Blues", axis=None))

In [ ]:
# ── Figure 3 · the whole business in one picture ─────────────────────────
# Colour = category identity (the validated FTW/APP/ACC palette), so a reader can
# see at a glance that the entire non-apparel business is the short tail.
plot = tree.sort_values("revenue_mn", ascending=False)
labels = [f"{r.sub_category}" for r in plot.itertuples()]
colors = [CAT_COLOR[r.category] for r in plot.itertuples()]
vals   = plot.pct_of_total.tolist()
yy     = range(len(plot))

fig, ax = plt.subplots(figsize=(11.5, 0.42 * len(plot) + 2.0))
# Tall figure: without this the default top margin leaves an inch of dead space
# between the legend and the first bar.
fig.subplots_adjust(top=0.98, bottom=0.05)
ax.barh(yy, vals, height=0.66, color=colors, zorder=3)
for i, v in enumerate(vals):
    ax.text(v + max(vals) * 0.012, i, f"{v:.2f}%", va="center", ha="left",
            fontsize=10, color=INK, fontweight="bold" if v >= 5 else "normal", zorder=4)

ax.set_yticks(list(yy)); ax.set_yticklabels(labels, color=INK, fontsize=11)
ax.set_xlim(0, max(vals) * 1.13)
ax.xaxis.set_major_formatter(mticker.PercentFormatter())
ax.invert_yaxis(); strip_frame(ax)
ax.grid(axis="x", color=GRID, lw=1, zorder=0); ax.set_axisbelow(True)

from matplotlib.patches import Patch
fig.legend(handles=[Patch(facecolor=CAT_COLOR[c], label=CAT_LABEL[c]) for c in CAT_ORDER],
           loc="upper left", bbox_to_anchor=(0.008, 1 + 0.13 / fig.get_size_inches()[1]),
           ncol=3, frameon=False, fontsize=11)
figh = fig.get_size_inches()[1]
fig.suptitle("Where the business comes from — % of total store sales",
             x=0.008, ha="left", y=1 + 0.88 / figh, fontsize=15, fontweight="bold")
fig.text(0.008, 1 + 0.38 / figh,
         "Every sub-category across all three categories. Vietnam, Thailand and China combined.",
         ha="left", fontsize=10.5, color=INK_MUTED)
plt.show()

In [ ]:
# --- What the tree says, in words ----------------------------------------
top3 = tree.nlargest(3, "pct_of_total")
non_app = tree[tree.category != "APP"]
big_ftw = tree[tree.category == "FTW"].nlargest(1, "pct_of_total").iloc[0]
big_acc = tree[tree.category == "ACC"].nlargest(1, "pct_of_total").iloc[0]

print("READING THE TREE")
print(f"  \u2022 The top 3 sub-categories are {', '.join(top3.sub_category)} \u2014 "
      f"{top3.pct_of_total.sum():.1f}% of the entire business between them.")
print(f"  \u2022 Everything outside apparel adds up to {non_app.pct_of_total.sum():.1f}% "
      f"across {len(non_app)} sub-categories.")
print(f"  \u2022 Footwear is concentrated: {big_ftw.sub_category} alone is "
      f"{big_ftw.pct_of_category:.0f}% of footwear.")
print(f"  \u2022 Accessories are concentrated too: {big_acc.sub_category} is "
      f"{big_acc.pct_of_category:.0f}% of accessories \u2014 the category is essentially a "
      f"{big_acc.sub_category.lower()} business.")
tail = tree[tree.pct_of_total < 0.5]
print(f"  \u2022 {len(tail)} of {len(tree)} sub-categories are each under 0.5% of sales, "
      f"{tail.pct_of_total.sum():.1f}% combined \u2014 that is where assortment gets thin.")

### 3.2 · Inside apparel — the sub-category split

Apparel is ~90% of the business, so it gets its own read below: the mix within apparel, how
each market skews against the three-market blend, and breadth-vs-return per sub-category.

*The third level of the tree — contribution by **size** — is in **section 4.4**, since it
needs the size query that section 4 builds.*

In [ ]:
# Raw string: sub-category classification relies entirely on \b regex word
# boundaries, so this must stay raw or every row falls through to the ELSE.
SUB_CTE   = r"""
, apparel AS (
  SELECT
    *
  FROM classified
  WHERE category = 'APP'
)
"""
Q_SUBCAT  = BASE_CTE + SUB_CTE + """
-- @tag: subcat
SELECT
  country,
  sub_category,
  COUNT(DISTINCT product_code)                                  AS styles,
  ROUND(SUM(revenue)/1e6, 1)                                    AS revenue_mn_vnd,
  ROUND(100 * SUM(revenue)
        / SUM(SUM(revenue)) OVER (PARTITION BY country), 1)     AS pct_of_app,
  CAST(SUM(units_sold) AS INT64)                                AS units,
  ROUND(100 * SUM(units_sold)
        / SUM(SUM(units_sold)) OVER (PARTITION BY country), 1)  AS pct_of_app_units,
  ROUND(SAFE_DIVIDE(SUM(revenue), SUM(units_sold))/1e6, 2)      AS aur_mn_vnd
FROM apparel
GROUP BY country, sub_category
ORDER BY country, revenue_mn_vnd DESC
"""

subcat = run_query(Q_SUBCAT)
subcat["country"] = pd.Categorical(subcat["country"], COUNTRY_ORDER, ordered=True)

# --- Reconciliation: sub-categories must sum back to the APP totals already sent ---
recon = (subcat.groupby("country", observed=True)["revenue_mn_vnd"].sum()
         - share[share.category == "APP"].set_index("country")["revenue_mn_vnd"])
assert recon.abs().max() < 1.0, f"sub-categories do not reconcile to APP: {recon.to_dict()}"
print(f"PASS \u00b7 sub-categories reconcile to APP totals "
      f"(max diff {recon.abs().max():.2f}mn VND, rounding only)")

SUB_ORDER = (subcat.groupby("sub_category", observed=True)["revenue_mn_vnd"]
                   .sum().sort_values(ascending=False).index.tolist())

sub_pct = (subcat.pivot(index="sub_category", columns="country", values="pct_of_app")
                 .reindex(SUB_ORDER))
print("\nAPPAREL MIX \u2014 % of each market's APPAREL sales, YTD Jan\u2013Jul 2026")

display(sub_pct.style.format("{:.1f}%", na_rep="\u2013")                     .background_gradient(cmap="Blues", axis=None))

In [ ]:
# --- Where each market skews vs the 3-market blend ------------------------
blend = (subcat.groupby("sub_category", observed=True)["revenue_mn_vnd"].sum()
         / subcat["revenue_mn_vnd"].sum() * 100).rename("blend_pct")

# .columns is a CategoricalIndex (inherited from subcat["country"]); .join() on
# that dtype raises InvalidIndexError in pandas, so flatten to plain strings first.
skew = sub_pct.copy()
skew.columns = skew.columns.astype(str)
skew = skew.join(blend).reindex(SUB_ORDER)
for ctry in rows:
    skew[f"{ctry.title()} idx"] = (skew[ctry] / skew["blend_pct"]).round(2)

print("MIX INDEX vs the 3-market blend  (1.00 = in line; >1 over-indexes)")
display(skew[["blend_pct"] + [f"{c.title()} idx" for c in rows]]
        .rename(columns={"blend_pct": "Blend %"})
        .style.format({"Blend %": "{:.1f}%", **{f"{c.title()} idx": "{:.2f}" for c in rows}},
                      na_rep="–")
        .background_gradient(cmap="RdYlGn", vmin=0.4, vmax=1.6,
                             subset=[f"{c.title()} idx" for c in rows]))


In [ ]:
# --- Full detail incl. AUR and breadth -----------------------------------
sub_detail = subcat.rename(columns={
    "country": "Market", "sub_category": "Sub-category", "styles": "Styles",
    "revenue_mn_vnd": "Revenue (mn VND)", "pct_of_app": "% of apparel",
    "units": "Units", "pct_of_app_units": "% of apparel units", "aur_mn_vnd": "AUR (mn VND)",
}).sort_values(["Market", "Revenue (mn VND)"], ascending=[True, False])

display(sub_detail.set_index(["Market", "Sub-category"]).style.format({
    "Styles": "{:,.0f}", "Revenue (mn VND)": "{:,.1f}", "% of apparel": "{:.1f}%",
    "Units": "{:,.0f}", "% of apparel units": "{:.1f}%", "AUR (mn VND)": "{:,.2f}",
}))

In [ ]:
# ── Figure 4 · apparel mix, small multiples ───────────────────────────────
# One series per panel, so identity comes from the panel title and a single hue
# is correct here — the category palette stays reserved for FTW/APP/ACC.
BLUE = CAT_COLOR["APP"]
plot_order = SUB_ORDER[::-1]                      # largest at top after invert
yy = range(len(plot_order))
xmax = float(sub_pct.max().max()) * 1.28

fig, axes = plt.subplots(1, 3, figsize=(14.5, 5.6), sharex=True)
fig.subplots_adjust(wspace=0.08)

for ax, ctry in zip(axes, COUNTRY_ORDER):
    raw  = [sub_pct.loc[s, ctry] for s in plot_order]
    vals = [0.0 if pd.isna(v) else v for v in raw]
    ax.barh(yy, vals, height=0.62, color=BLUE, zorder=3)
    for i, (v, r) in enumerate(zip(vals, raw)):
        # "–" means the market sold none of it; "0.0%" means it sold some that
        # rounds below 0.05% — a real distinction when planning a buy.
        lab = "–" if pd.isna(r) else f"{v:.1f}%"
        ax.text(v + xmax * 0.015, i, lab, va="center", ha="left", fontsize=9.5,
                color=INK_MUTED if pd.isna(r) else INK,
                fontweight="bold" if v >= 10 else "normal", zorder=4)
    ax.set_title(ctry.title(), loc="left", fontsize=12.5, fontweight="bold", pad=8)
    ax.set_xlim(0, xmax); ax.set_yticks(list(yy))
    ax.xaxis.set_major_formatter(mticker.PercentFormatter())
    ax.grid(axis="x", color=GRID, lw=1, zorder=0); ax.set_axisbelow(True)
    strip_frame(ax)

axes[0].set_yticklabels(plot_order, color=INK, fontsize=11)
for ax in axes[1:]:
    ax.set_yticklabels([])

fig.suptitle("What apparel is made of — % of each market's apparel sales",
             x=0.008, ha="left", y=1.10, fontsize=15, fontweight="bold")
fig.text(0.008, 1.035,
         "Sub-category derived from product name. Jan–Jul 2026, stores only.",
         ha="left", fontsize=10.5, color=INK_MUTED)
plt.show()

## 4 · Size curve — how to split the buy by size

> *"Previously also ask for your share of biz % by product size in each sub category type,
> eg Dress 35% small, 45% Medium 20% Large. To use as reference to help with buying the
> right sizing for each sub category product."* — Nigel

**Measured in units, not value.** A size curve is a buying ratio, and an OTB is placed in
units — weighting by revenue would let an expensive size masquerade as a popular one.

`fn_dim_sku()` supplies `size_label` with a `size_order` sort key, so sizes sort XS → XL
rather than alphabetically. Coverage is strong: **97.7% of apparel units sold carry a real
alpha size**; the remaining 2.3% are FREESIZE, which have no size decision to make and are
excluded from the curve rather than silently diluting it.

In [ ]:
Q_SIZE = BASE_CTE + """
-- @tag: size
SELECT
  country,
  category,
  sub_category,
  COALESCE(size, '(unsized)')     AS size,
  COALESCE(size_type, 'N/A')      AS size_type,   -- ALPHA / NUMERIC / ONE SIZE / N/A
  size_order,
  CAST(SUM(units_sold) AS INT64)  AS units,
  ROUND(SUM(revenue)/1e6, 1)      AS revenue_mn_vnd
FROM classified
WHERE category IS NOT NULL          -- every size system kept; filtered per section below
GROUP BY country, category, sub_category, size, size_type, size_order
ORDER BY country, category, sub_category, size_order
"""

size = run_query(Q_SIZE)
SIZED = ("ALPHA", "NUMERIC")        # ONE SIZE / N/A carry no buying decision

app = size[(size.category == "APP") & (size.size_type == "ALPHA")]

# Derive from APPAREL only: ACC carries a stray ALPHA size ("SMALL", on bags) that
# would otherwise land in the apparel size order and shift every colour step.
ALPHA_ORDER = app.sort_values("size_order")["size"].drop_duplicates().tolist()
print("Apparel sizes in size_order:", " < ".join(ALPHA_ORDER))


def size_curve(df, index_cols, order):
    """Rows = index_cols, columns = sizes, values = % of that row's units."""
    piv = (df.pivot_table(index=index_cols, columns="size", values="units",
                          aggfunc="sum", observed=True)
             .reindex(columns=order))
    return piv.div(piv.sum(axis=1), axis=0).mul(100).round(1)


from matplotlib.colors import LinearSegmentedColormap, to_rgb


def _lum(c):
    """WCAG relative luminance — decides whether a label sits in ink or white."""
    f = lambda v: v / 12.92 if v <= 0.04045 else ((v + 0.055) / 1.055) ** 2.4
    r, g, b = to_rgb(c)
    return 0.2126 * f(r) + 0.7152 * f(g) + 0.0722 * f(b)


def make_ramp(labels, light, dark):
    """Sequential ramp for an ORDERED dimension (size), sampled to fit however many
    steps the period actually contains — a fixed colour list breaks when a size is
    absent. Returns {label: colour} and the set of steps light enough for dark ink."""
    cmap = LinearSegmentedColormap.from_list("ramp", [light, dark])
    n = max(len(labels) - 1, 1)
    ramp = {s: cmap(i / n) for i, s in enumerate(labels)}
    return ramp, {s for s in labels if _lum(ramp[s]) > 0.20}


def stacked_curve_chart(curve_df, order, ramp, light_steps, title, subtitle,
                        min_label=6):
    """100% stacked horizontal bars — one row per sub-category, segments = sizes."""
    plot_rows = list(curve_df.index)
    yy = range(len(plot_rows))
    fig, ax = plt.subplots(figsize=(12.5, 0.52 * len(plot_rows) + 2.1))
    left = [0.0] * len(plot_rows)

    for s in order:
        vals = [float(curve_df.loc[r, s]) if pd.notna(curve_df.loc[r, s]) else 0.0
                for r in plot_rows]
        ax.barh(yy, vals, left=left, height=0.62, color=ramp[s],
                edgecolor=SURFACE, linewidth=2, zorder=3, label=str(s))
        for i, (v, l) in enumerate(zip(vals, left)):
            if v >= min_label:
                ax.text(l + v / 2, i, f"{v:.0f}%", ha="center", va="center",
                        fontsize=10, fontweight="bold", zorder=4,
                        color=INK if s in light_steps else "white")
        left = [l + v for l, v in zip(left, vals)]

    ax.set_yticks(list(yy)); ax.set_yticklabels(plot_rows, color=INK, fontsize=11.5)
    ax.set_xlim(0, 100); ax.set_xticks([0, 25, 50, 75, 100])
    ax.xaxis.set_major_formatter(mticker.PercentFormatter())
    ax.invert_yaxis(); strip_frame(ax)
    ax.grid(axis="x", color=GRID, lw=1, zorder=0); ax.set_axisbelow(True)

    # Title block offsets are held in INCHES and converted, so a 3-row chart gets the
    # same visual spacing as a 12-row one. Fixed figure-fraction values collide as
    # soon as the figure gets shorter.
    figh = fig.get_size_inches()[1]
    y = lambda inches: 1 + inches / figh
    handles, labels_ = ax.get_legend_handles_labels()
    fig.legend(handles, labels_, loc="upper left", bbox_to_anchor=(0.008, y(0.13)),
               ncol=len(order), frameon=False, fontsize=11)
    fig.suptitle(title, x=0.008, ha="left", y=y(0.88), fontsize=15, fontweight="bold")
    fig.text(0.008, y(0.38), subtitle, ha="left", fontsize=10.5, color=INK_MUTED)
    plt.show()

In [ ]:
# ── The answer, in Nigel's format: one curve per apparel sub-category ─────
MIN_UNITS = 100            # below this a curve is noise, not a signal
vol = app.groupby("sub_category", observed=True)["units"].sum().sort_values(ascending=False)
keep = vol[vol >= MIN_UNITS].index.tolist()
thin = vol[vol < MIN_UNITS]

curve = size_curve(app[app.sub_category.isin(keep)], "sub_category", ALPHA_ORDER).loc[keep]
curve.insert(0, "Units", vol[keep].astype(int))

print("APPAREL SIZE CURVE \u2014 % of units by size, all 3 markets, YTD Jan\u2013Jul 2026")
display(curve.style.format({**{s: "{:.1f}%" for s in ALPHA_ORDER}, "Units": "{:,.0f}"},
                           na_rep="\u2013")
             .background_gradient(cmap="Blues", axis=1, subset=ALPHA_ORDER))

if len(thin):
    print(f"Held back for thin volume (<{MIN_UNITS} units): "
          + ", ".join(f"{k} ({v:,.0f}u)" for k, v in thin.items()))

In [ ]:
# ── Does the curve travel? Same question, split by market ────────────────
by_mkt = size_curve(app, "country", ALPHA_ORDER).reindex(COUNTRY_ORDER)
by_mkt.insert(0, "Units", app.groupby("country", observed=True)["units"].sum()
                              .reindex(COUNTRY_ORDER).astype(int))

print("APPAREL SIZE CURVE BY MARKET \u2014 % of units")
display(by_mkt.style.format({**{s: "{:.1f}%" for s in ALPHA_ORDER}, "Units": "{:,.0f}"},
                            na_rep="\u2013")
              .background_gradient(cmap="Blues", axis=1, subset=ALPHA_ORDER))

# Blended curve across all three markets — reused by the read-out in section 7.
blend_curve = app.groupby("size", observed=True)["units"].sum().reindex(ALPHA_ORDER)
blend_curve = (blend_curve / blend_curve.sum() * 100).round(1)
BLEND_TEXT  = " / ".join(f"{s} {blend_curve[s]:.0f}%" for s in ALPHA_ORDER)

spread = (by_mkt[ALPHA_ORDER].max() - by_mkt[ALPHA_ORDER].min()).round(1)
print("\nWidest gap between markets, by size (percentage points):")
print(spread.to_string())
print(f"\nMax spread across all sizes: {spread.max():.1f}pp \u2014 "
      + ("the curve is effectively market-independent; one buy ratio can serve all three."
         if spread.max() <= 5 else
         "big enough to warrant a per-market curve."))

In [ ]:
# ── Figure 5 · apparel size mix by sub-category ──────────────────────────
# Size is an ORDERED dimension, so colour is sequential (one hue, light→dark),
# not categorical. The ramp encodes XS→XL; it does not encode magnitude.
RAMP, LIGHT_STEPS = make_ramp(ALPHA_ORDER, "#DCE6F5", "#1B4A94")   # apparel = brand blue

stacked_curve_chart(
    curve[ALPHA_ORDER], ALPHA_ORDER, RAMP, LIGHT_STEPS,
    "Size curve by apparel sub-category — % of units",
    "All three markets combined. Segments under 6% are left unlabelled.")

### 4.1 · A caution before anyone buys to this curve

**What sold is not the same as what was wanted.** A size curve built from sales is censored
by what was on the shelf: if S sold out in week three, S sales understate S demand, and the
curve quietly tells you to buy too few of the size you keep running out of.

The check below compares the size mix of *sales* against the size mix of *stock still on
hand*, and the sell-through rate each size achieved. If the small sizes sell through hardest
and the large sizes are what's left over, the buy has been running too big — and the true
demand curve is even more S-weighted than the sales curve shows.

*Scope caveat: `remain_quantity` is stock across all channels, while sales here are stores
only, so treat the sell-through level as indicative. The **ranking between sizes** is the
reliable part, and that is what the decision turns on.*

In [ ]:
Q_SIZE_ST = """
-- @tag: size_st
WITH dim AS (
  SELECT DISTINCT
    s.variation_code, category, size_label, size_type, size_order, sell_through_rate,sl_ton_kho
  FROM `{project}.reportdb.fn_dim_sku`() s
  LEFT JOIN `{project}.reportdb.report_stats_product` sp ON s.variation_code = sp.variation_code
),
sold AS (
  SELECT variation_code, SUM(units_sold) AS units
  FROM `{project}.reportdb.report_order_product_revenue_summary`
  WHERE period_date BETWEEN @d_start AND @d_end
    AND country IN UNNEST(@countries)
    AND level_2 LIKE @channel_pattern
  GROUP BY variation_code
)
SELECT
  d.size_label                       AS size,
  MIN(d.size_order)                  AS size_order,
  CAST(SUM(s.units) AS INT64)        AS sold_units,
  CAST(SUM(sl_ton_kho) AS INT64)     AS remain_units,
  AVG(sell_through_rate)             AS sell_through_rate
FROM dim d
JOIN sold s USING (variation_code)
WHERE d.category = 'APP' AND d.size_type = 'ALPHA'
GROUP BY d.size_label
ORDER BY size_order
"""

st = run_query(Q_SIZE_ST).set_index("size").reindex(ALPHA_ORDER)
st["pct_of_sales"]  = (st.sold_units   / st.sold_units.sum()   * 100).round(1)
st["pct_of_stock"]  = (st.remain_units / (st.remain_units.sum()) * 100).round(1)
st["sell_through"]  = (st.sell_through_rate * 100).round(1)
st["gap_pp"]        = (st.pct_of_sales - st.pct_of_stock).round(1)

print("SIZE: WHAT SOLD vs WHAT IS LEFT")
display(st[["sold_units", "pct_of_sales", "remain_units", "pct_of_stock",
            "sell_through", "gap_pp"]]
        .rename(columns={"sold_units": "Sold", "pct_of_sales": "% of sales",
                         "remain_units": "Stock left", "pct_of_stock": "% of stock",
                         "sell_through": "Sell-through", "gap_pp": "Sales − stock (pp)"})
        .style.format({"Sold": "{:,.0f}", "% of sales": "{:.1f}%", "Stock left": "{:,.0f}",
                       "% of stock": "{:.1f}%", "Sell-through": "{:.1f}%",
                       "Sales − stock (pp)": "{:+.1f}"})
        .background_gradient(cmap="RdYlGn", subset=["Sell-through"])
        .background_gradient(cmap="RdYlGn", subset=["Sales − stock (pp)"]))

best, worst = st.sell_through.idxmax(), st.sell_through.idxmin()
print(f"\nSell-through is highest on {best} ({st.loc[best,'sell_through']:.1f}%) and "
      f"lowest on {worst} ({st.loc[worst,'sell_through']:.1f}%).")
over = st[st.gap_pp < 0].index.tolist()
if over:
    print(f"Sizes over-represented in leftover stock vs demand: {', '.join(over)} "
          f"\u2014 the buy has been running too big.")

In [ ]:
# ── Figure 6 · sell-through by size ──────────────────────────────────────
fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 3.6),
                             gridspec_kw={"width_ratios": [1, 1.15], "wspace": 0.28})

# Left: sell-through rate — one series, so no legend; the title names it.
a1.bar(ALPHA_ORDER, st.sell_through, width=0.6,
       color=[RAMP[s] for s in ALPHA_ORDER], zorder=3)
for i, s in enumerate(ALPHA_ORDER):
    a1.text(i, st.loc[s, "sell_through"] + 1.2, f"{st.loc[s,'sell_through']:.0f}%",
            ha="center", fontsize=10.5, fontweight="bold", color=INK, zorder=4)
a1.set_ylim(0, max(st.sell_through) * 1.28)
a1.yaxis.set_major_formatter(mticker.PercentFormatter())
a1.grid(axis="y", color=GRID, lw=1, zorder=0); a1.set_axisbelow(True); strip_frame(a1)
a1.set_title("Sell-through by size", loc="left", fontsize=12.5, fontweight="bold", pad=10)

# Right: demand vs leftover, as a gap — diverging around zero.
GOOD, BAD = "#1F4E9C", "#C2410C"
gaps = st.gap_pp.tolist()
a2.barh(range(len(ALPHA_ORDER)), gaps, height=0.55,
        color=[GOOD if g >= 0 else BAD for g in gaps], zorder=3)
for i, g in enumerate(gaps):
    a2.text(g + (0.6 if g >= 0 else -0.6), i, f"{g:+.1f}pp", va="center",
            ha="left" if g >= 0 else "right", fontsize=10.5, fontweight="bold",
            color=INK, zorder=4)
a2.axvline(0, color=INK, lw=1.2, zorder=5)
a2.set_yticks(range(len(ALPHA_ORDER))); a2.set_yticklabels(ALPHA_ORDER, fontsize=11)
lim = max(abs(min(gaps)), abs(max(gaps))) * 1.5
a2.set_xlim(-lim, lim); a2.invert_yaxis()
a2.grid(axis="x", color=GRID, lw=1, zorder=0); a2.set_axisbelow(True); strip_frame(a2)
a2.set_title("Share of sales minus share of leftover stock", loc="left",
             fontsize=12.5, fontweight="bold", pad=10)

fig.suptitle("Are we buying the right sizes?", x=0.008, ha="left", y=1.20,
             fontsize=15, fontweight="bold")
fig.text(0.008, 1.09,
         "Right panel: positive = sells faster than it was bought (under-bought). "
         "Negative = piling up (over-bought).",
         ha="left", fontsize=10.5, color=INK_MUTED)
plt.show()

### 4.2 · Footwear — a different size system

Footwear runs on EU numeric sizing (36–40 in the trading period), so it needs its own curve;
it cannot be blended into the apparel one. Coverage is **100%** — every footwear unit sold
carries a size.

The caveat is volume, not coverage. Footwear is a small part of the business, so these curves
sit on hundreds of units rather than tens of thousands. They are a reasonable starting point,
not the settled answer the apparel curve is.

In [ ]:
ftw = size[(size.category == "FTW") & (size.size_type == "NUMERIC")].copy()
NUM_ORDER = ftw.sort_values("size_order")["size"].drop_duplicates().tolist()
print("Footwear sizes in size_order:", " < ".join(NUM_ORDER))

FTW_MIN = 50
ftw_vol  = ftw.groupby("sub_category", observed=True)["units"].sum().sort_values(ascending=False)
ftw_keep = ftw_vol[ftw_vol >= FTW_MIN].index.tolist()
ftw_thin = ftw_vol[ftw_vol < FTW_MIN]

ftw_curve = size_curve(ftw[ftw.sub_category.isin(ftw_keep)], "sub_category", NUM_ORDER).loc[ftw_keep]
ftw_show  = ftw_curve.copy()
ftw_show.insert(0, "Units", ftw_vol[ftw_keep].astype(int))

print(f"\nFOOTWEAR SIZE CURVE \u2014 % of units (sub-categories with \u2265{FTW_MIN} units)")
display(ftw_show.style.format({**{s: "{:.1f}%" for s in NUM_ORDER}, "Units": "{:,.0f}"},
                              na_rep="\u2013")
                .background_gradient(cmap="Oranges", axis=1, subset=NUM_ORDER))
if len(ftw_thin):
    print(f"Held back for thin volume (<{FTW_MIN} units): "
          + ", ".join(f"{k} ({v:,.0f}u)" for k, v in ftw_thin.items()))

ftw_mkt = size_curve(ftw, "country", NUM_ORDER).reindex(COUNTRY_ORDER)
ftw_mkt.insert(0, "Units", ftw.groupby("country", observed=True)["units"].sum()
                              .reindex(COUNTRY_ORDER).astype(int))
print("\nFOOTWEAR SIZE CURVE BY MARKET \u2014 % of units")
display(ftw_mkt.style.format({**{s: "{:.1f}%" for s in NUM_ORDER}, "Units": "{:,.0f}"},
                             na_rep="\u2013")
               .background_gradient(cmap="Oranges", axis=1, subset=NUM_ORDER))

ftw_spread = (ftw_mkt[NUM_ORDER].max() - ftw_mkt[NUM_ORDER].min()).round(1)
print(f"\nMax spread between markets: {ftw_spread.max():.1f}pp \u2014 "
      + ("wider than apparel, so footwear may warrant a per-market curve"
         if ftw_spread.max() > spread.max() else
         "no wider than apparel") +
      f". But the whole category is only {ftw['units'].sum():,.0f} units over 7 months, so read\n"
      f"these as directional. The apparel curve is the one with enough volume to buy on.")

In [ ]:
# ── Figure 7 · footwear size mix ─────────────────────────────────────────
# Orange ramp, matching footwear's slot in the category palette — so a reader never
# confuses this chart's encoding with the apparel one above.
FTW_RAMP, FTW_LIGHT = make_ramp(NUM_ORDER, "#FDE6D2", "#C25E08")

stacked_curve_chart(
    ftw_curve[NUM_ORDER], NUM_ORDER, FTW_RAMP, FTW_LIGHT,
    "Size curve by footwear sub-category — % of units",
    f"EU sizing. All three markets combined; only sub-categories with \u2265{FTW_MIN} units shown.")

### 4.3 · Which sub-categories actually have a size decision

Before Nigel asks why there's no accessories size curve: for most of accessories, there isn't
one to give. A bag, a pair of sunglasses, a wallet or a scarf has no size — that is a property
of the product, not a gap in the data.

The table below is the honest coverage map across **every** category. It answers "can I plan a
size buy here?" in one column, and it is worth reading before anyone goes looking for a curve
that cannot exist.

In [ ]:
# ── Size coverage across every category ──────────────────────────────────
# Built from plain groupby aggregations rather than .apply(): the apply form needs
# pandas' include_groups argument, which is not stable across versions.
KEYS = ["category", "sub_category"]
tot = size.groupby(KEYS, observed=True).agg(
    units=("units", "sum"), revenue_mn=("revenue_mn_vnd", "sum"))

sized_only = size[size.size_type.isin(SIZED)]
szd = sized_only.groupby(KEYS, observed=True).agg(
    sized_units=("units", "sum"),
    n_sizes=("size", "nunique"),
    size_system=("size_type", lambda s: "/".join(sorted(set(s)))))

cov = tot.join(szd, how="left").reset_index()
cov[["sized_units", "n_sizes"]] = cov[["sized_units", "n_sizes"]].fillna(0)
cov["size_system"] = cov["size_system"].fillna("none")
cov["pct_sized"] = (100 * cov.sized_units / cov.units.clip(lower=1)).round(1)
cov["revenue_mn"] = cov.revenue_mn.round(1)
cov = cov.astype({"units": int, "n_sizes": int})
cov = cov.sort_values(["category", "units"], ascending=[True, False])


def verdict(r):
    """A curve needs a real size system, more than one size, and enough units."""
    if r.pct_sized < 50:            return "No size — nothing to plan"
    if r.n_sizes < 2:               return "Single size only — no curve"
    if r.units < 100:               return "Too thin to buy on"
    return "Curve usable"


cov["verdict"] = cov.apply(verdict, axis=1)

STYLE = {"Curve usable": "background-color:#E3F0E6;color:#14532D",
         "Too thin to buy on": "background-color:#FDF0DC;color:#7C3A03",
         "Single size only — no curve": "background-color:#F2F2F0;color:#57534E",
         "No size — nothing to plan": "background-color:#F2F2F0;color:#57534E"}

print("SIZE COVERAGE \u2014 can we plan a size buy for this sub-category?")
display(cov.set_index(["category", "sub_category"])
           .rename(columns={"units": "Units", "revenue_mn": "Revenue (mn VND)",
                            "pct_sized": "% of units sized", "size_system": "Size system",
                            "n_sizes": "Sizes", "verdict": "Verdict"})
           .drop(columns=["sized_units"])
           .style.format({"Units": "{:,.0f}", "Revenue (mn VND)": "{:,.1f}",
                          "% of units sized": "{:.1f}%", "Sizes": "{:,.0f}"})
           .apply(lambda col: [STYLE.get(v, "") for v in col], subset=["Verdict"]))

usable = cov[cov.verdict == "Curve usable"]
print(f"\n{len(usable)} of {len(cov)} sub-categories support a size curve, covering "
      f"{100*usable['units'].sum()/cov['units'].sum():.1f}% of all units sold.")

for cat in ["ACC", "APP", "FTW"]:
    c = cov[cov.category == cat]
    if not len(c):
        continue
    print(f"  {cat}: {100*c.loc[c.verdict=='Curve usable','units'].sum()/max(c['units'].sum(),1):.0f}% "
          f"of units sit in a sub-category with a usable curve")

In [ ]:
# --- The accessories answer, stated plainly -------------------------------
acc = size[size.category == "ACC"]
acc_sized = acc[acc.size_type.isin(SIZED)]

print("ACCESSORIES \u2014 SIZE POSITION")
print(f"  {acc['units'].sum():,.0f} units sold across all three markets.")
print(f"  {100*acc_sized['units'].sum()/max(acc['units'].sum(),1):.0f}% carry any size at all, "
      f"and those use {acc_sized['size'].nunique()} distinct size label(s): "
      f"{', '.join(sorted(acc_sized['size'].unique())) or 'none'}.")
print("  Bags, sunglasses, wallets and scarves are unsized by nature \u2014 there is no size")
print("  curve to give, and none is needed to place the accessories buy.")

# Reused by the read-out in section 7.
acc_unsized = 100 * (1 - acc_sized["units"].sum() / max(acc["units"].sum(), 1))

display(acc.groupby(["sub_category", "size", "size_type"], observed=True)
           .agg(units=("units", "sum"), revenue_mn=("revenue_mn_vnd", "sum"))
           .sort_values("units", ascending=False))

# One apparel sub-category is also sizeless — worth catching before it surprises a buyer.
freesize = size[(size.category == "APP") & (size.size_type == "ONE SIZE")]
if len(freesize):
    fs = freesize.groupby("sub_category", observed=True)["units"].sum().sort_values(ascending=False)
    print("\nAlso note \u2014 FREESIZE apparel (no size decision either):")
    for k, v in fs.items():
        tot = size[(size.category == "APP") & (size.sub_category == k)]["units"].sum()
        print(f"  {k}: {v:,.0f} of {tot:,.0f} units ({100*v/tot:.0f}%) are one-size")

### 4.4 · Size contribution — sizes as a share of the whole business

Sections 4 to 4.2 answer *"how do I split this sub-category across sizes"* — a curve that
sums to 100% within its own row. This closes the loop with the other question: **which sizes
are the business?** Every size is sized against total store sales, so it nests directly under
the category and sub-category levels in section 3.1 and the whole thing sums to 100%.

Unsized product (bags, sunglasses, FREESIZE knitwear) is kept in as its own row rather than
dropped, so the denominator is the real business and nothing is quietly hidden.

In [ ]:
# ── Level 3 of the contribution tree: size ───────────────────────────────
tot_u, tot_r = size["units"].sum(), size["revenue_mn_vnd"].sum()

# dropna=False: a NULL size_type would otherwise be dropped from the group keys and
# quietly shrink the denominator of a table that is asserted to sum to 100%.
sz = (size.groupby(["category", "size", "size_type"], observed=True, dropna=False)
          .agg(units=("units", "sum"), revenue_mn=("revenue_mn_vnd", "sum"),
               order=("size_order", "min"))
          .reset_index())
sz["pct_of_total_sales"] = (100 * sz.revenue_mn / tot_r).round(2)
sz["pct_of_total_units"] = (100 * sz.units / tot_u).round(2)
sz["pct_of_cat_units"] = (100 * sz.units
                          / sz.groupby("category", observed=True)["units"]
                              .transform("sum")).round(1)
sz["category"] = pd.Categorical(sz["category"], CAT_ORDER, ordered=True)
sz = sz.sort_values(["category", "order"])

print("LEVEL 3 \u2014 CONTRIBUTION BY SIZE, all 3 markets, YTD Jan\u2013Jul 2026")
display(sz.assign(Category=sz.category.map(CAT_LABEL))
          .set_index(["Category", "size"])
          .drop(columns=["category", "order"])
          .rename(columns={"size_type": "System", "units": "Units",
                           "revenue_mn": "Revenue (mn VND)",
                           "pct_of_total_sales": "% of TOTAL sales",
                           "pct_of_total_units": "% of TOTAL units",
                           "pct_of_cat_units": "% of own category units"})
          .style.format({"Units": "{:,.0f}", "Revenue (mn VND)": "{:,.1f}",
                         "% of TOTAL sales": "{:.2f}%", "% of TOTAL units": "{:.2f}%",
                         "% of own category units": "{:.1f}%"})
          .background_gradient(cmap="Blues", subset=["% of TOTAL units"]))

assert abs(sz.pct_of_total_units.sum() - 100) < 0.3, "size rows do not sum to 100% of units"
print(f"\nRows sum to {sz.pct_of_total_units.sum():.1f}% of units and "
      f"{sz.pct_of_total_sales.sum():.1f}% of sales \u2014 the whole business is accounted for.")

In [ ]:
# --- The same view per market --------------------------------------------
mkt_sz = (size.groupby(["country", "category", "size"], observed=True)["units"].sum()
              .reset_index())
mkt_sz["pct"] = (100 * mkt_sz.units
                 / mkt_sz.groupby("country", observed=True)["units"].transform("sum")).round(2)

order_ix = [(c, s) for c in CAT_ORDER
            for s in sz[sz.category == c].sort_values("order")["size"]]
piv = (mkt_sz.pivot_table(index=["category", "size"], columns="country",
                          values="pct", aggfunc="sum", observed=True)
             .reindex(columns=COUNTRY_ORDER)
             .reindex(order_ix))
piv.index = pd.MultiIndex.from_tuples([(CAT_LABEL[c], s) for c, s in piv.index],
                                      names=["Category", "size"])

print("CONTRIBUTION BY SIZE, PER MARKET \u2014 % of that market's total store units")
display(piv.style.format("{:.2f}%", na_rep="\u2013")
                 .background_gradient(cmap="Blues", axis=None))

In [ ]:
# ── Figure 8 · which sizes are the business ──────────────────────────────
# Colour = category (the validated FTW/APP/ACC palette); the x-axis carries the
# magnitude, so the ordered nature of size is left to the axis labels.
plot_sz = sz[sz.units > 0].copy()
labels = [f"{CAT_LABEL[r.category]} · {r.size}" for r in plot_sz.itertuples()]
colors = [CAT_COLOR[r.category] for r in plot_sz.itertuples()]
vals   = plot_sz.pct_of_total_units.tolist()
yy     = range(len(plot_sz))

fig, ax = plt.subplots(figsize=(11, 0.40 * len(plot_sz) + 2.0))
fig.subplots_adjust(top=0.98, bottom=0.06)
ax.barh(yy, vals, height=0.64, color=colors, zorder=3)
for i, v in enumerate(vals):
    ax.text(v + max(vals) * 0.012, i, f"{v:.2f}%", va="center", ha="left",
            fontsize=10, color=INK, fontweight="bold" if v >= 10 else "normal", zorder=4)

ax.set_yticks(list(yy)); ax.set_yticklabels(labels, color=INK, fontsize=10.5)
ax.set_xlim(0, max(vals) * 1.14)
ax.xaxis.set_major_formatter(mticker.PercentFormatter())
ax.invert_yaxis(); strip_frame(ax)
ax.grid(axis="x", color=GRID, lw=1, zorder=0); ax.set_axisbelow(True)

from matplotlib.patches import Patch
figh = fig.get_size_inches()[1]
fig.legend(handles=[Patch(facecolor=CAT_COLOR[c], label=CAT_LABEL[c]) for c in CAT_ORDER],
           loc="upper left", bbox_to_anchor=(0.008, 1 + 0.13 / figh),
           ncol=3, frameon=False, fontsize=11)
fig.suptitle("Which sizes are the business — % of total store units",
             x=0.008, ha="left", y=1 + 0.88 / figh, fontsize=15, fontweight="bold")
fig.text(0.008, 1 + 0.38 / figh,
         "All three markets combined. Unsized product kept in, so the bars sum to 100%.",
         ha="left", fontsize=10.5, color=INK_MUTED)
plt.show()

In [ ]:
# --- What the size level says --------------------------------------------
app_sz = sz[(sz.category == "APP") & (sz.size_type == "ALPHA")]
top_sz = app_sz.nlargest(1, "pct_of_total_units").iloc[0]
sm_share = app_sz[app_sz["size"].isin(["XS", "S"])].pct_of_total_units.sum()
big_share = app_sz[app_sz["size"].isin(["L", "XL"])].pct_of_total_units.sum()
unsized_share = sz[~sz.size_type.isin(SIZED)].pct_of_total_units.sum()

print("READING THE SIZE LEVEL")
print(f"  \u2022 {CAT_LABEL['APP']} {top_sz['size']} alone is {top_sz.pct_of_total_units:.1f}% of every "
      f"unit the stores sell \u2014 the single biggest line in the business.")
print(f"  \u2022 Small end (XS+S) is {sm_share:.1f}% of all store units; "
      f"large end (L+XL) is {big_share:.1f}%. That is a {sm_share/max(big_share,0.01):.1f}:1 ratio.")
print(f"  \u2022 Footwear across every size is {sz[sz.category=='FTW'].pct_of_total_units.sum():.1f}% "
      f"of units; accessories {sz[sz.category=='ACC'].pct_of_total_units.sum():.1f}%.")
print(f"  \u2022 {unsized_share:.1f}% of units carry no size at all (bags, sunglasses, FREESIZE).")
print("\n  Practical read: a size plan that treats S and M as the core, with L/XL as")
print("  service sizes rather than volume sizes, matches how the business actually sells.")

## 5 · Is it moving?

A 7-month average hides direction. Accessories is a young category at LSOUL — if it is
ramping, the YTD average understates what an opening buy should carry, and the exit rate
is the better planning number.

In [ ]:
Q_TREND = BASE_CTE + """
-- @tag: trend
, monthly AS (
  SELECT
    DATE_TRUNC(period_date, MONTH) AS month_start,
    country,
    category,
    SUM(revenue)    AS revenue,
    SUM(units_sold) AS units
  FROM classified
  WHERE category IN ('FTW','APP','ACC')
  GROUP BY month_start, country, category
)
SELECT
  FORMAT_DATE('%Y-%m', month_start)                             AS month,
  country,
  category,
  ROUND(100 * revenue
        / SUM(revenue) OVER (PARTITION BY country, month_start), 1) AS pct_of_sales,
  ROUND(revenue/1e6, 1)                                         AS revenue_mn_vnd
FROM monthly
ORDER BY country, month, category
"""

trend = run_query(Q_TREND)

# Months with zero sales in a category return no row — fill them so lines don't jump.
grid_ix = pd.MultiIndex.from_product(
    [sorted(trend["month"].unique()), COUNTRY_ORDER, CAT_ORDER],
    names=["month", "country", "category"])
trend = (trend.set_index(["month", "country", "category"])
              .reindex(grid_ix)
              .fillna({"pct_of_sales": 0.0, "revenue_mn_vnd": 0.0})
              .reset_index())

pivot_trend = trend.pivot_table(index=["country", "month"],
                                columns="category", values="pct_of_sales")
display(pivot_trend.style.format("{:.1f}%"))

In [ ]:
months = sorted(trend["month"].unique())
xs     = range(len(months))

fig, axes = plt.subplots(1, 3, figsize=(14, 3.9), sharey=True)
fig.subplots_adjust(wspace=0.12)

for ax, ctry in zip(axes, COUNTRY_ORDER):
    d = trend[trend["country"] == ctry]
    for cat in ["FTW", "ACC"]:
        vals = [d[(d["month"] == m) & (d["category"] == cat)]["pct_of_sales"].sum()
                for m in months]
        ax.plot(xs, vals, color=CAT_COLOR[cat], lw=2, marker="o", ms=5.5,
                mec=SURFACE, mew=1.6, zorder=3, label=CAT_LABEL[cat])
        ax.annotate(f"{vals[-1]:.1f}%", (len(months) - 1, vals[-1]),
                    textcoords="offset points", xytext=(7, 0), va="center",
                    fontsize=10, fontweight="bold", color=INK)

    ax.set_title(ctry.title(), loc="left", fontsize=12.5, fontweight="bold", pad=8)
    ax.set_xticks(list(xs)); ax.set_xticklabels([m[-2:] for m in months], fontsize=10)
    ax.set_xlim(-0.3, len(months) - 0.3 + 0.9)
    ax.set_ylim(0, 14); ax.yaxis.set_major_formatter(mticker.PercentFormatter())
    ax.grid(axis="y", color=GRID, lw=1, zorder=0); ax.set_axisbelow(True)
    strip_frame(ax)

axes[0].set_xlabel("month of 2026", fontsize=10)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper left", bbox_to_anchor=(0.008, 1.08),
           ncol=2, frameon=False, fontsize=11)
fig.suptitle("Footwear & accessories share by month — stores",
             x=0.008, ha="left", y=1.36, fontsize=15, fontweight="bold")
fig.text(0.008, 1.22, "Apparel is the residual and is omitted so the small categories stay readable.",
         ha="left", fontsize=10.5, color=INK_MUTED)
plt.show()

## 6 · Is low accessories share a demand problem or an assortment problem?

This is the section that actually answers Nigel's OTB question, and it is the one the raw
share table cannot answer on its own.

A category can under-index on sales for two very different reasons, with opposite buying implications:

- **We don't carry enough of it** → the fix is to *buy more*
- **We carry it and it doesn't sell** → the fix is to *buy less, and buy better*

The **space-to-sales index** separates them: `% of sales ÷ % of styles carried`.
Above 1.0 the category earns more than its share of the assortment; below 1.0 it earns less.

In [ ]:
Q_PROD = BASE_CTE + """
-- @tag: productivity
SELECT
  country,
  category,
  COUNT(DISTINCT product_code)                                  AS styles,
  ROUND(100 * COUNT(DISTINCT product_code)
        / SUM(COUNT(DISTINCT product_code))
              OVER (PARTITION BY country), 1)                   AS pct_of_styles,
  ROUND(100 * SUM(revenue)
        / SUM(SUM(revenue)) OVER (PARTITION BY country), 1)     AS pct_of_sales,
  ROUND(SAFE_DIVIDE(SUM(units_sold), COUNT(DISTINCT product_code)), 1)      AS units_per_style,
  ROUND(SAFE_DIVIDE(SUM(revenue), COUNT(DISTINCT product_code))/1e6, 1)     AS revenue_mn_per_style
FROM classified
WHERE category IN ('FTW','APP','ACC')
GROUP BY country, category
ORDER BY country, pct_of_sales DESC
"""

prod = run_query(Q_PROD)
prod["index_s2s"] = (prod["pct_of_sales"] / prod["pct_of_styles"]).round(2)
prod["category"]  = pd.Categorical(prod["category"], CAT_ORDER, ordered=True)
prod["country"]   = pd.Categorical(prod["country"],  COUNTRY_ORDER, ordered=True)
prod = prod.sort_values(["country", "category"])

view = prod.rename(columns={
    "country": "Market", "category": "Cat", "styles": "Styles",
    "pct_of_styles": "% of styles", "pct_of_sales": "% of sales",
    "units_per_style": "Units/style", "revenue_mn_per_style": "Rev mn/style",
    "index_s2s": "Space-to-sales index",
})
view["Cat"] = view["Cat"].map(CAT_LABEL)
display(view.set_index(["Market", "Cat"]).style.format({
    "Styles": "{:,.0f}", "% of styles": "{:.1f}%", "% of sales": "{:.1f}%",
    "Units/style": "{:,.1f}", "Rev mn/style": "{:,.1f}", "Space-to-sales index": "{:.2f}",
}).background_gradient(cmap="RdYlGn", subset=["Space-to-sales index"], vmin=0, vmax=1.5))

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.0))
h = 0.24

for j, cat in enumerate(CAT_ORDER):
    d    = prod[prod["category"] == cat].set_index("country")
    vals = [d.loc[r, "index_s2s"] for r in rows]
    offs = [i + (j - 1) * h for i in y]
    ax.barh(offs, vals, height=h * 0.84, color=CAT_COLOR[cat], zorder=3,
            label=CAT_LABEL[cat])
    for off, v in zip(offs, vals):
        ax.text(v + 0.02, off, f"{v:.2f}", va="center", ha="left",
                fontsize=10.5, color=INK, fontweight="bold", zorder=4)

ax.axvline(1.0, color=INK, lw=1.6, ls=(0, (4, 3)), zorder=5)
ax.annotate("1.00 — earns exactly its\nshare of the assortment",
            xy=(1.0, -0.62), xytext=(1.06, -0.62), fontsize=9.5,
            color=INK_MUTED, va="center", annotation_clip=False)

ax.set_yticks(list(y)); ax.set_yticklabels([r.title() for r in rows], color=INK, fontsize=12)
ax.set_xlim(0, 1.42); ax.invert_yaxis(); strip_frame(ax)
ax.grid(axis="x", color=GRID, lw=1, zorder=0); ax.set_axisbelow(True)

handles, labels = ax.get_legend_handles_labels()
fig.legend(handles, labels, loc="upper left", bbox_to_anchor=(0.008, 1.06),
           ncol=3, frameon=False, fontsize=11)
fig.suptitle("Space-to-sales index — % of sales ÷ % of styles carried",
             x=0.008, ha="left", y=1.29, fontsize=15, fontweight="bold")
fig.text(0.008, 1.17, "Below 1.00 = the category takes more of the buy than it returns in sales.",
         ha="left", fontsize=10.5, color=INK_MUTED)
plt.show()

## 7 · Read-out for Nigel

The cell below assembles the reply from the queried figures rather than from hard-coded
numbers, so it stays correct if the period or market set is changed at the top.

In [ ]:
st_S,  st_XL  = st.loc["S", "sell_through"], st.loc["XL", "sell_through"]
gap_L, gap_XL = abs(st.loc["L", "gap_pp"]), abs(st.loc["XL", "gap_pp"])
pct_S = st.loc["S", "pct_of_sales"]


def pct(ctry, cat, frame=pct_sales):
    return frame.loc[ctry, cat]


def aur(ctry, cat):
    return share[(share.country == ctry) & (share.category == cat)]["aur_mn_vnd"].iloc[0]

app_lo = min(pct(c, "APP") for c in rows)
app_hi = max(pct(c, "APP") for c in rows)
app_top = max(rows, key=lambda c: pct(c, "APP")).title()
app_top_v = max(pct(c, "APP") for c in rows)
ftw_top = max(rows, key=lambda c: pct(c, "FTW")).title()
ftw_top_v = max(pct(c, "FTW") for c in rows)
acc_top = max(rows, key=lambda c: pct(c, "ACC")).title()
acc_top_v = max(pct(c, "ACC") for c in rows)

lo  = min(pct(c, "ACC") for c in rows)
hi  = max(pct(c, "ACC") for c in rows)
u_lo = min(pct_units.loc[c, "ACC"] for c in rows)
u_hi = max(pct_units.loc[c, "ACC"] for c in rows)

exit_acc = {c: trend[(trend.country == c) & (trend.category == "ACC") &
                     (trend.month == months[-1])]["pct_of_sales"].sum() for c in rows}
s2s_acc  = {c: prod[(prod.country == c) & (prod.category == "ACC")]["index_s2s"].iloc[0]
            for c in rows}

print(f"""
SHARE OF BUSINESS — LSOUL STORES, YTD JAN–JUL 2026
══════════════════════════════════════════════════════════════════════

               Apparel        Footwear      Accessories
""" + "\n".join(
    f"  {c.title():<11}  {pct(c,'APP'):>5.1f}%         {pct(c,'FTW'):>4.1f}%          {pct(c,'ACC'):>4.1f}%"
    for c in rows) + f"""

1. APPAREL IS THE BUSINESS — {app_lo:.1f}–{app_hi:.1f}% of store sales in every market.
   Any OTB split that doesn't start from ~{app_lo:.0f}% apparel is not a split, it's a bet.

2. THE "LOT OF ACCESSORIES" PREMISE DOESN'T HOLD ON SELL-OUT.
   Accessories are {lo:.1f}–{hi:.1f}% of sales and only {u_lo:.1f}–{u_hi:.1f}% of units.
   What is true is that we CARRY a lot of accessory styles relative to what they return:
   space-to-sales index of {min(s2s_acc.values()):.2f}–{max(s2s_acc.values()):.2f} (1.00 = fair share).
   That's an over-assortment signal, not a demand signal — the read is
   BUY NARROWER AND DEEPER in accessories, not bigger.

3. BUT DON'T PLAN OFF THE 7-MONTH AVERAGE ALONE — ACCESSORIES ARE RAMPING.
   July exit rate: """ + " · ".join(f"{c.title()} {exit_acc[c]:.1f}%" for c in rows) + f"""
   vs a YTD average of {lo:.1f}–{hi:.1f}%. Every market exits above its own average,
   so plan the opening nearer the exit rate than the 7-month mean.

4. SPLIT THE OTB IN UNITS, NOT VALUE — THEY ARE NOT THE SAME.
   Footwear is {pct('VIETNAM','FTW'):.1f}% of Vietnam sales but only {pct_units.loc['VIETNAM','FTW']:.1f}% of units,
   because footwear AUR ({aur('VIETNAM','FTW'):.2f}mn VND) runs {aur('VIETNAM','FTW')/aur('VIETNAM','APP'):.1f}x apparel ({aur('VIETNAM','APP'):.2f}mn).
   Buying to the value split would over-buy footwear units by roughly
   {pct('VIETNAM','FTW')/pct_units.loc['VIETNAM','FTW']:.1f}x.

5. MARKET NUANCE. {ftw_top} leads on footwear ({ftw_top_v:.1f}%) and {acc_top} on
   accessories ({acc_top_v:.1f}%); {app_top} is the most apparel-dependent ({app_top_v:.1f}%).
   If the opening is in one specific market, use that market's column, not the average.

6. INSIDE APPAREL, THREE SUB-CATEGORIES ARE THE WHOLE BUY.
   Dresses + Tops + Skirts = """ + " · ".join(
        f"{c.title()} {sub_pct.loc[['Dresses','Tops','Skirts'], c].sum():.0f}%"
        for c in rows) + f""" of apparel.
   The lead differs by market though: Vietnam is TOPS-led ({sub_pct.loc['Tops','VIETNAM']:.1f}% vs
   dresses {sub_pct.loc['Dresses','VIETNAM']:.1f}%), while Thailand and China are DRESS-led
   ({sub_pct.loc['Dresses','THAILAND']:.1f}% and {sub_pct.loc['Dresses','CHINA']:.1f}%).
   Vietnam also over-indexes on trousers ({sub_pct.loc['Trousers','VIETNAM']:.1f}% vs
   {sub_pct.loc['Trousers','CHINA']:.1f}% China); Thailand over-indexes on sets
   ({sub_pct.loc['Sets / Co-ords','THAILAND']:.1f}% vs {sub_pct.loc['Sets / Co-ords','VIETNAM']:.1f}% Vietnam).

7. WATCH THE LABELS IF ANYONE BUYS OFF THE RAW PRODUCT TREE.
   This notebook now reads sub-category from fn_dim_sku(), which is clean. The underlying
   product_tree_csv is NOT: its 'QUAN JEAN' bucket is ~97% non-denim trousers, and its
   'SUIT' bucket holds only bodysuits. True denim is {sub_pct.loc['Denim'].mean():.1f}% of apparel.
   Anyone self-serving off the raw tree would order denim and suits we don't sell.

8. SIZE: BUY SMALL. S IS THE BUSINESS, NOT M.
   Blended apparel curve is {BLEND_TEXT}.
   The curve barely moves between markets (max {spread.max():.1f}pp gap on any one size),
   so a single apparel buy ratio serves Vietnam, Thailand and China alike.
   Footwear is the exception: EU sizing, and a {ftw_spread.max():.1f}pp spread between markets,
   so size it per market. Accessories need no size plan at all - {acc_unsized:.0f}% of
   accessory units are unsized by nature (bags, sunglasses, wallets, scarves).

9. AND THE CURRENT BUY IS RUNNING TOO BIG.
   S sells through at {st_S:.0f}% but XL at only {st_XL:.0f}%.
   L and XL take up {gap_L:.1f}pp and {gap_XL:.1f}pp more of leftover stock than of demand.
   Because S sells out first, the sales curve UNDERSTATES S - treat {pct_S:.0f}% S as a
   FLOOR for the opening buy, not a target.

CAVEAT: China is a single location and Thailand's Central Ladprao opened part-way through
the period, so those mixes sit on a thinner base than Vietnam's.
""")

## 8 · Export

One tidy sheet to attach to the WhatsApp reply.

In [ ]:
export = (share.merge(prod[["country", "category", "styles", "pct_of_styles", "index_s2s"]],
                      on=["country", "category"], how="left")
               .sort_values(["country", "category"]))
export["category"] = export["category"].map(CAT_LABEL)
export = export.rename(columns={
    "country": "market", "category": "category",
    "revenue_mn_vnd": "revenue_mn_vnd", "pct_of_sales": "pct_of_sales",
    "units": "units", "pct_of_units": "pct_of_units", "aur_mn_vnd": "aur_mn_vnd",
    "styles": "styles_carried", "pct_of_styles": "pct_of_styles",
    "index_s2s": "space_to_sales_index",
})

sub_export = subcat.rename(columns={"country": "market"}).sort_values(
    ["market", "revenue_mn_vnd"], ascending=[True, False])

OUT     = f"lsoul_category_share_{DATE_START}_{DATE_END}.csv"
OUT_SUB = f"lsoul_apparel_subcategory_{DATE_START}_{DATE_END}.csv"
export.to_csv(OUT, index=False)
sub_export.to_csv(OUT_SUB, index=False)
print(f"Written: {OUT}")
print(f"Written: {OUT_SUB}")
size_export = size.copy()
OUT_SIZE = f"lsoul_size_curve_{DATE_START}_{DATE_END}.csv"
size_export.to_csv(OUT_SIZE, index=False)
print(f"Written: {OUT_SIZE}")

display(export)
display(curve)
display(sub_export)

---
*Generated from `reportdb.report_order_product_revenue_summary` joined to `reportdb.product_tree_csv`.
Revenue is net and VND-normalised. Re-run any time — all parameters live in section 1.*